# FairFace Ethical AI Demo: Biased vs Unbiased Training Data

**Goal:** Demonstrate how *representation bias* in training data can create **unequal performance** across demographic groups, and how training on the original (unbiased) data can reduce the harm.

**Task in this notebook:** Predict **gender** (Male/Female) from face images using a **ResNet50** classifier, then evaluate fairness by:
- **Race**
- **Race × Gender** (intersectional groups)

We will train:
- **Model A (Biased)** → trained on an intentionally skewed subset (`biased_train`)
- **Model U (Unbiased)** → trained on the original training data (`train_unbiased`)

> Note: This is a *fairness demo*. In real governance work, you should also consider consent, privacy, purpose limitation, and legal/organizational policy.


## 0) Setup & Imports


In [ ]:
import os, glob
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

print("Torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1) Dataset paths (FairFace margin025)


In [ ]:
# Kaggle dataset root (update if your dataset slug differs)
DATASET_DIR = "/kaggle/input/fairface-aml"

# Choose padding variant (you said you're using margin025)
IMAGE_ROOT = os.path.join(DATASET_DIR, "fairface-img-margin025-trainval")

LABELS_CSV = os.path.join(DATASET_DIR, "fairface_label_train.csv")

print("DATASET_DIR:", DATASET_DIR)
print("IMAGE_ROOT :", IMAGE_ROOT, "exists=", os.path.exists(IMAGE_ROOT))
print("LABELS_CSV :", LABELS_CSV, "exists=", os.path.exists(LABELS_CSV))

# Quick listing to help debug paths
if os.path.exists(DATASET_DIR):
    print("\nTop-level folders:")
    for p in sorted(glob.glob(DATASET_DIR + "/*"))[:20]:
        print(" -", os.path.basename(p))


## 2) Load labels and basic sanity checks


In [ ]:
labels = pd.read_csv(LABELS_CSV)

# Make sure race/gender are strings
labels["race"] = labels["race"].astype(str)
labels["gender"] = labels["gender"].astype(str)

print("Rows:", len(labels))
display(labels.head())

print("\nColumns:", list(labels.columns))
print("\nRace counts (train+test in this CSV):")
display(labels["race"].value_counts().head(10))


## 3) Split into trainval vs test using `service_test`


In [ ]:
trainval = labels[labels["service_test"] == False].copy()
test = labels[labels["service_test"] == True].copy()

print("trainval:", len(trainval))
print("test    :", len(test))

# Test should be balanced across race in FairFace (good for fairness evaluation)
display(test["race"].value_counts().head(10))


## 4) Create a balanced validation set (by race) and a clean training pool


In [ ]:
val_per_race = 500  # adjust up/down for compute budget

val_parts = []
for race in trainval["race"].unique():
    sub = trainval[trainval["race"] == race]
    take = min(val_per_race, len(sub))
    val_parts.append(sub.sample(n=take, random_state=123))

val = pd.concat(val_parts).reset_index(drop=True)

# Train pool excludes validation to avoid leakage
train_pool = trainval[~trainval["file"].isin(val["file"])].copy()

print("val:", len(val))
print("train_pool:", len(train_pool))

# Leakage check (should be 0)
overlap = set(train_pool["file"]).intersection(set(val["file"]))
print("train_pool ∩ val overlap:", len(overlap))


## 5) Create the **unbiased** training set (original data)


In [ ]:
# Unbiased/original training data = full train_pool (no artificial downsampling)
train_unbiased = train_pool.copy()

print("train_unbiased:", len(train_unbiased))
print("\nUnbiased train race distribution (proportions):")
display(train_unbiased["race"].value_counts(normalize=True).round(4))


## 6) Create the **biased** training set (intersectional caps: race × gender)
This is where we intentionally create unfair representation.


In [ ]:
# Clean gender into consistent values
train_pool["gender_clean"] = train_pool["gender"].astype(str).str.strip().str.title()

caps = {
    ("White","Male"): 12000,
    ("East Asian","Female"): 12000,

    ("White","Female"): 8000,
    ("East Asian","Male"): 8000,

    ("Black","Male"): 50,
    ("Black","Female"): 20,
    ("Indian","Male"): 2000,
    ("Indian","Female"): 200,
    ("Southeast Asian","Male"): 2000,
    ("Southeast Asian","Female"): 200,
    ("Middle Eastern","Male"): 2000,
    ("Middle Eastern","Female"): 200,
    ("Latino_Hispanic","Male"): 50,
    ("Latino_Hispanic","Female"): 20,
}


parts = []
for (race, gender), cap in caps.items():
    sub = train_pool[(train_pool["race"] == race) & (train_pool["gender_clean"] == gender)]
    take = min(cap, len(sub))
    if take > 0:
        parts.append(sub.sample(n=take, random_state=42))

biased_train = pd.concat(parts).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("biased_train:", len(biased_train))

ct = pd.crosstab(biased_train["race"], biased_train["gender_clean"])
ct_norm = pd.crosstab(biased_train["race"], biased_train["gender_clean"], normalize=True)

print("\nBiased TRAIN distribution (counts):")
display(ct)

print("\nBiased TRAIN distribution (%):")
display((ct_norm*100).round(2))


### Visualize: biased vs unbiased training distributions (race × gender)


In [ ]:
# Ensure gender_clean exists for unbiased train
train_unbiased["gender_clean"] = train_unbiased["gender"].astype(str).str.strip().str.title()

ct_biased = pd.crosstab(biased_train["race"], biased_train["gender_clean"])
ct_unbias = pd.crosstab(train_unbiased["race"], train_unbiased["gender_clean"])

# Align rows/cols
all_races = sorted(set(ct_biased.index).union(ct_unbias.index))
all_genders = sorted(set(ct_biased.columns).union(ct_unbias.columns))
ct_biased = ct_biased.reindex(index=all_races, columns=all_genders, fill_value=0)
ct_unbias = ct_unbias.reindex(index=all_races, columns=all_genders, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

ct_biased.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set_title("Biased training data (Model A)", fontsize=14, pad=10)
axes[0].set_xlabel("Race"); axes[0].set_ylabel("Number of images")
axes[0].tick_params(axis="x", labelrotation=35)

ct_unbias.plot(kind="bar", stacked=True, ax=axes[1], width=0.85)
axes[1].set_title("Unbiased/original training data (Model U)", fontsize=14, pad=10)
axes[1].set_xlabel("Race")
axes[1].tick_params(axis="x", labelrotation=35)

axes[1].legend_.remove()
axes[0].legend(title="Gender", fontsize=11, title_fontsize=11, loc="upper right")

plt.tight_layout()
plt.show()


## 7) Labels (`y`) and a robust image loader


In [ ]:
# -----------------------------
# Race × Gender (14-class) target creation
# -----------------------------
CLASS_NAMES = [
    "Black_Female",
    "White_Female",
    "East_Asian_Male",
    "Middle_Eastern_Female",
    "Indian_Female",
    "Southeast_Asian_Male",
    "Latino_Hispanic_Female",
    "Southeast_Asian_Female",
    "Indian_Male",
    "White_Male",
    "Latino_Hispanic_Male",
    "Black_Male",
    "East_Asian_Female",
    "Middle_Eastern_Male",
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# FairFace race strings -> canonical strings used above
RACE_CANON = {
    "Black": "Black",
    "White": "White",
    "Indian": "Indian",
    "East Asian": "East_Asian",
    "Southeast Asian": "Southeast_Asian",
    "Middle Eastern": "Middle_Eastern",
    "Latino_Hispanic": "Latino_Hispanic",
    "Latino Hispanic": "Latino_Hispanic",
}

def clean_gender(g: str) -> str:
    g = str(g).strip().lower()
    if g == "male":
        return "Male"
    if g == "female":
        return "Female"
    return "Unknown"

def clean_race(r: str) -> str:
    r = str(r).strip()
    return RACE_CANON.get(r, r.replace(" ", "_"))

def add_y(df):
    df = df.copy()

    df["gender_clean"] = df["gender"].apply(clean_gender)
    df["race_clean"] = df["race"].apply(clean_race)

    # Combined label must match CLASS_NAMES exactly
    df["combined"] = df["race_clean"] + "_" + df["gender_clean"]

    # Map to integer class index 0..NUM_CLASSES-1
    df["y"] = df["combined"].map(CLASS_TO_IDX)

    unknown = df[df["y"].isna()]["combined"].value_counts()
    if len(unknown) > 0:
        print("⚠️ Unknown combined labels not in CLASS_NAMES (fix mapping / class list):")
        print(unknown.head(50))

    return df

biased_train = add_y(biased_train)
train_unbiased = add_y(train_unbiased)
val = add_y(val)
test = add_y(test)

# Drop rows that didn't map (should be 0 if mappings match)
biased_train = biased_train.dropna(subset=["y"]).copy()
train_unbiased = train_unbiased.dropna(subset=["y"]).copy()
val = val.dropna(subset=["y"]).copy()
test = test.dropna(subset=["y"]).copy()

# Ensure integer labels
biased_train["y"] = biased_train["y"].astype(int)
train_unbiased["y"] = train_unbiased["y"].astype(int)
val["y"] = val["y"].astype(int)
test["y"] = test["y"].astype(int)

print("Unknown y (biased_train):", biased_train["y"].isna().sum())
print("Unknown y (train_unbiased):", train_unbiased["y"].isna().sum())
print("Unknown y (val):", val["y"].isna().sum())
print("Unknown y (test):", test["y"].isna().sum())

# Quick sanity checks (prevents CUDA device-side asserts)
print("NUM_CLASSES:", NUM_CLASSES)
print("biased_train y min/max:", int(biased_train["y"].min()), int(biased_train["y"].max()))
print("val y min/max:", int(val["y"].min()), int(val["y"].max()))
assert biased_train["y"].min() >= 0 and biased_train["y"].max() < NUM_CLASSES
assert val["y"].min() >= 0 and val["y"].max() < NUM_CLASSES


## 8) Dataset & transforms


In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset

class FairFaceDataset(Dataset):
    def __init__(self, df, image_root, transform=None):
        if "y" not in df.columns:
            raise ValueError("DataFrame missing 'y'. Run add_y() before creating Dataset.")
        self.df = df.dropna(subset=["y"]).reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # FairFace CSV uses "train/xxx.jpg", "val/xxx.jpg"
        img_path = os.path.join(self.image_root, row["file"])

        img = Image.open(img_path).convert("RGB")
        y = torch.tensor(int(row["y"]), dtype=torch.long)

        if self.transform is not None:
            img = self.transform(img)

        return img, y


In [ ]:
from torchvision import transforms

# Simple transforms (good baseline)
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


## 9) DataLoaders


In [ ]:
BATCH_SIZE = 128  # you tested this works on Kaggle GPU
NUM_WORKERS = 2

# Biased (Model A)
trainA_ds = FairFaceDataset(biased_train, IMAGE_ROOT, transform=train_tfms)

# Unbiased (Model U)
trainU_ds = FairFaceDataset(train_unbiased, IMAGE_ROOT, transform=train_tfms)

# Shared val/test
val_ds = FairFaceDataset(val, IMAGE_ROOT, transform=val_tfms)
test_ds = FairFaceDataset(test, IMAGE_ROOT, transform=val_tfms)

trainA_loader = DataLoader(trainA_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
trainU_loader = DataLoader(trainU_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Batches (trainA, trainU, val, test):", len(trainA_loader), len(trainU_loader), len(val_loader), len(test_loader))


## 10) Model factory + training helpers


In [ ]:
# -----------------------------
# ResNet50 for Race × Gender (multi-class)
# -----------------------------
import time

def build_resnet50(num_classes=NUM_CLASSES, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    m = models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)                # [B, NUM_CLASSES]
        loss = criterion(logits, y)      # CrossEntropyLoss expects y in [0..NUM_CLASSES-1]

        loss_sum += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return loss_sum / max(total, 1), correct / max(total, 1)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return loss_sum / max(total, 1), correct / max(total, 1)

@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds_all = []
    for x, _y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        preds_all.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds_all) if preds_all else np.array([], dtype=np.int64)

@torch.no_grad()
def predict_topk(model, x, k=5):
    """Top-k predictions for a batch tensor x (useful for the web app later)."""
    model.eval()
    x = x.to(device, non_blocking=True)

    t0 = time.perf_counter()
    logits = model(x)
    probs = torch.softmax(logits, dim=1)
    inference_ms = (time.perf_counter() - t0) * 1000.0

    top_p, top_i = torch.topk(probs, k=min(k, probs.size(1)), dim=1)

    topk = []
    for b in range(x.size(0)):
        items = []
        for p, idx in zip(top_p[b].tolist(), top_i[b].tolist()):
            label = IDX_TO_CLASS[idx] if "IDX_TO_CLASS" in globals() else str(idx)
            items.append({"label": label, "p": float(p)})
        topk.append(items)

    return topk, inference_ms


## 11) Train **Model A (Biased)**


In [ ]:
import copy

modelA = build_resnet50(num_classes=NUM_CLASSES).to(device)

criterionA = nn.CrossEntropyLoss()
optimizerA = torch.optim.AdamW(modelA.parameters(), lr=3e-4, weight_decay=1e-4)

MAX_EPOCHS = 50      # hard cap (so it doesn't run forever)
PATIENCE = 5         # stop if no val improvement for this many epochs
MIN_DELTA = 1e-4     # minimum improvement to count as "better"

best_val = -np.inf
best_epoch = 0
patience_left = PATIENCE

best_state = copy.deepcopy(modelA.state_dict())

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(modelA, trainA_loader, optimizerA, criterionA)
    va_loss, va_acc = evaluate(modelA, val_loader, criterionA)

    print(
        f"[Model A] Epoch {epoch:02d} | "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
        f"val loss={va_loss:.4f} acc={va_acc:.4f} | "
        f"best={best_val:.4f} (epoch {best_epoch}) | "
        f"patience_left={patience_left}"
    )

    # Check improvement
    if va_acc > best_val + MIN_DELTA:
        best_val = va_acc
        best_epoch = epoch
        patience_left = PATIENCE

        best_state = copy.deepcopy(modelA.state_dict())
        torch.save(best_state, "modelA_resnet50_biased.pth")
        print("✅ New best model saved.")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"⏹️ Early stopping triggered. No improvement for {PATIENCE} epochs.")
            break

# Restore best weights in memory
modelA.load_state_dict(best_state)
print(f"Best val acc (Model A): {best_val:.4f} at epoch {best_epoch}")
print("Best checkpoint saved at: modelA_resnet50_biased.pth")

## 12) Train **Model U (Unbiased)**


In [ ]:
import copy

modelU = build_resnet50(num_classes=NUM_CLASSES).to(device)

criterionU = nn.CrossEntropyLoss()
optimizerU = torch.optim.AdamW(modelU.parameters(), lr=3e-4, weight_decay=1e-4)

MAX_EPOCHS = 50      # hard cap
PATIENCE = 5         # stop if no improvement for this many epochs
MIN_DELTA = 1e-4     # minimum val-acc improvement to count

best_val_u = -np.inf
best_epoch_u = 0
patience_left = PATIENCE

best_state_u = copy.deepcopy(modelU.state_dict())

for epoch in range(1, MAX_EPOCHS + 1):
    # IMPORTANT: use the unbiased training loader here
    tr_loss, tr_acc = train_one_epoch(modelU, trainU_loader, optimizerU, criterionU)
    va_loss, va_acc = evaluate(modelU, val_loader, criterionU)

    print(
        f"[Model U] Epoch {epoch:02d} | "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
        f"val loss={va_loss:.4f} acc={va_acc:.4f} | "
        f"best={best_val_u:.4f} (epoch {best_epoch_u}) | "
        f"patience_left={patience_left}"
    )

    if va_acc > best_val_u + MIN_DELTA:
        best_val_u = va_acc
        best_epoch_u = epoch
        patience_left = PATIENCE

        best_state_u = copy.deepcopy(modelU.state_dict())
        torch.save(best_state_u, "modelU_resnet50_unbiased.pth")
        print("✅ New best Model U saved.")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"⏹️ Early stopping triggered for Model U. No improvement for {PATIENCE} epochs.")
            break

modelU.load_state_dict(best_state_u)
print(f"Best val acc (Model U): {best_val_u:.4f} at epoch {best_epoch_u}")
print("Best checkpoint saved at: modelU_resnet50_unbiased.pth")

## 13) Fairness evaluation (Race + Race×Gender)


In [ ]:
def fairness_report(model, checkpoint_path, test_df, test_loader, name="Model"):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    preds = predict_all(model, test_loader)

    te = test_df.dropna(subset=["y"]).reset_index(drop=True).copy()
    te["gender_clean"] = te["gender"].astype(str).str.strip().str.title()
    te["y_pred"] = preds

    overall = (te["y_pred"] == te["y"]).mean()

    # Race
    race_acc = te.groupby("race").apply(lambda g: (g["y_pred"] == g["y"]).mean())
    race_n = te["race"].value_counts()

    # Race × Gender
    acc_rg = te.groupby(["race","gender_clean"]).apply(lambda g: (g["y_pred"] == g["y"]).mean())
    n_rg = te.groupby(["race","gender_clean"]).size()

    print(f"\nOverall test accuracy ({name}):", overall)
    print(f"\nAccuracy by race ({name}):")
    display(pd.DataFrame({"acc": race_acc, "n": race_n}).sort_values("acc"))
    print("Worst-group accuracy:", float(race_acc.min()))
    print("Disparity gap (max-min):", float(race_acc.max() - race_acc.min()))

    print(f"\nAccuracy by race×gender ({name}):")
    display(pd.DataFrame({"acc": acc_rg, "n": n_rg}).sort_values("acc"))
    print("Worst intersectional group accuracy:", float(acc_rg.min()))
    print("Intersectional disparity gap (max-min):", float(acc_rg.max() - acc_rg.min()))

    return te, race_acc, acc_rg

test_eval_A, race_acc_A, acc_rg_A = fairness_report(modelA, "modelA_resnet50_biased.pth", test, test_loader, name="Model A")
test_eval_U, race_acc_U, acc_rg_U = fairness_report(modelU, "modelU_resnet50_unbiased.pth", test, test_loader, name="Model U")


## 14) Visual comparison: accuracy changes by intersectional group


In [ ]:
# Combine intersectional accuracies (race × gender) for both models
compare_rg = pd.concat(
    [
        acc_rg_A.rename("Model_A_biased"),
        acc_rg_U.rename("Model_U_unbiased"),
    ],
    axis=1
)

# Keep only groups that exist in BOTH series (optional; avoids NaNs in bars)
compare_rg = compare_rg.dropna(subset=["Model_A_biased", "Model_U_unbiased"])

compare_rg["delta_U_minus_A"] = compare_rg["Model_U_unbiased"] - compare_rg["Model_A_biased"]
compare_rg = compare_rg.sort_values("delta_U_minus_A")

display(compare_rg)

# Side-by-side bars
import numpy as np
x = np.arange(len(compare_rg))
w = 0.42
labels = [f"{r}/{g}" for (r, g) in compare_rg.index]

plt.figure(figsize=(16, 6))
plt.bar(x - w/2, compare_rg["Model_A_biased"], width=w, label="Model A (Biased)")
plt.bar(x + w/2, compare_rg["Model_U_unbiased"], width=w, label="Model U (Unbiased)")
plt.xticks(x, labels, rotation=70, ha="right")

# IMPORTANT: don't clip the y-axis, or low-accuracy groups will look like 0.
plt.ylim(0.0, 1.0)

plt.title("Test Accuracy by race×gender: Biased vs Unbiased training", fontsize=16, pad=12)
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Delta chart
plt.figure(figsize=(16, 5))
plt.bar(labels, compare_rg["delta_U_minus_A"])
plt.axhline(0, linewidth=1)
plt.xticks(rotation=70, ha="right")
plt.title("Fairness change (Model U - Model A) by race×gender", fontsize=16, pad=12)
plt.ylabel("Accuracy change")
plt.tight_layout()
plt.show()


## 15) Save artifacts for deployment
The `.pth` files appear in the notebook output and can be downloaded from Kaggle.
For Replit/Gradio, you'll copy these weights and your preprocessing settings.


In [ ]:
print("Saved checkpoints in current directory:")
print([p for p in os.listdir(".") if p.endswith(".pth")])

# In Kaggle, saving to /kaggle/working makes it easy to download from the Output panel.
# Uncomment if needed:
# !cp modelA_resnet50_biased.pth /kaggle/working/
# !cp modelU_resnet50_unbiased.pth /kaggle/working/


In [ ]:
import os, random, glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# -----------------------------
# SETTINGS
# -----------------------------
TEST_ROOT = "/kaggle/input/fairface-aml/test_images"  # your test image folders live here
N_IMAGES = 12
TOPK = 2   # ✅ show top 2
SEED = 7

# Make sure IDX_TO_CLASS exists
if "IDX_TO_CLASS" not in globals():
    IDX_TO_CLASS = {i: c for i, c in enumerate(CLASS_NAMES)}

def parse_race_gender(label: str):
    s = str(label).strip().replace("-", "_").replace(" ", "_")
    parts = [p for p in s.split("_") if p]
    gender = "Unknown"
    lp = [p.lower() for p in parts]
    if "male" in lp:
        gender = "Male"
        parts = [p for p in parts if p.lower() != "male"]
    elif "female" in lp:
        gender = "Female"
        parts = [p for p in parts if p.lower() != "female"]
    race = "_".join(parts) if parts else "Unknown"
    return race, gender

@torch.no_grad()
def predict_topk(model, pil_img, k=2):
    model.eval()
    x = val_tfms(pil_img).unsqueeze(0).to(device)  # reuse your val transforms
    logits = model(x)
    probs = F.softmax(logits, dim=1).squeeze(0).cpu()
    top_p, top_i = torch.topk(probs, k=min(k, probs.numel()))
    return [(IDX_TO_CLASS[int(i)], float(p)) for p, i in zip(top_p, top_i)]

def get_test_image_paths(root):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(root, "**", e), recursive=True))
    if len(paths) == 0:
        raise ValueError(f"No images found under: {root}")
    return paths

def show_predictions_for_model(model, model_name, paths, n_images=12, topk=2, seed=7):
    random.seed(seed)
    sample_paths = random.sample(paths, k=min(n_images, len(paths)))

    cols = 3
    rows = int(np.ceil(len(sample_paths) / cols))
    plt.figure(figsize=(6 * cols, 5 * rows))

    for idx, p in enumerate(sample_paths, start=1):
        img = Image.open(p).convert("RGB")
        preds = predict_topk(model, img, k=topk)

        folder_race = os.path.basename(os.path.dirname(p))  # e.g., Black

        # Build a nice multi-line prediction summary
        pred_lines = []
        for rank, (lab, prob) in enumerate(preds, start=1):
            race, gender = parse_race_gender(lab)
            pred_lines.append(f"#{rank}: {lab} ({prob:.2f}) → {race}, {gender}")

        title = (
            f"{model_name}\n"
            f"Folder: {folder_race} | File: {os.path.basename(p)}\n"
            + "\n".join(pred_lines)
        )

        ax = plt.subplot(rows, cols, idx)
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# -----------------------------
# RUN: once for each model
# -----------------------------
paths = get_test_image_paths(TEST_ROOT)

show_predictions_for_model(modelA, "Model A (Biased)", paths, n_images=N_IMAGES, topk=TOPK, seed=SEED)
show_predictions_for_model(modelU, "Model U (Unbiased)", paths, n_images=N_IMAGES, topk=TOPK, seed=SEED)
